# Modelo preditivo de risco de defasagem

## Objetivo

Este notebook treina e avalia um modelo **realmente preditivo** para estimar o **risco de defasagem futura** do aluno, usando os indicadores observados no ano atual.

Nesta versão, a base de modelagem já foi ajustada para trabalhar com o alvo futuro:

- **`defasagem_ano_seguinte`**
- **`risco_defasagem_futuro`**

Assim, o problema de Machine Learning passa a ser:

> Dado o desempenho e o contexto do aluno no ano atual, prever a chance de ele apresentar defasagem no ano seguinte.


## Estratégia metodológica

Para tornar a modelagem mais consistente, este notebook adota as seguintes decisões:

- uso da base processada `base_processada_reduzida_ML.parquet`;
- preparação das variáveis com as funções do novo `src/features.py`;
- pipeline com imputação, padronização e codificação categórica;
- **regressão logística** como modelo baseline;
- **validação temporal preferencial**, usando o último ano disponível como teste, para respeitar a natureza preditiva do problema.

Caso a separação temporal não seja possível por falta de anos suficientes com alvo válido, o notebook usa divisão aleatória estratificada como fallback.


## Por que a regressão logística foi escolhida inicialmente

A regressão logística foi utilizada como primeiro modelo por motivos importantes para esta etapa do projeto:

- **Interpretabilidade**: permite entender com clareza como os atributos influenciam o risco previsto.
- **Baseline robusto**: é um modelo clássico e confiável para comparação com modelos mais complexos.
- **Boa adequação para classificação binária**: o alvo principal é prever `risco_defasagem_futuro` (0 ou 1).
- **Facilidade de deploy no Streamlit**: é leve, rápida e simples de serializar e publicar.


### 1) Importações e configuração do ambiente


In [1]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().resolve().parents[0] if "notebooks" in str(Path.cwd()) else Path.cwd().resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.features import (
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN,
    prepare_training_features,
)
from src.model import (
    load_training_data,
    build_pipeline,
    MODEL_FILE,
    METADATA_FILE,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

print("ROOT_DIR:", ROOT_DIR)
print("TARGET_COLUMN:", TARGET_COLUMN)
print("FEATURE_COLUMNS:", FEATURE_COLUMNS)

ROOT_DIR: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos
TARGET_COLUMN: risco_defasagem_futuro
FEATURE_COLUMNS: ['inde', 'n_av', 'iaa', 'ieg', 'ips', 'ipp', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian', 'fase_ideal']


In [2]:
from src.model import train_model

artifacts = train_model()

print("Classes:", artifacts["model"].named_steps["model"].classes_)

Arquivo de treino: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/processed/base_processada_reduzida_ML.parquet
Classes do modelo: [0 1]
Pipeline salvo em: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/artifacts/model_pipeline.joblib
Metadados salvos em: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/artifacts/model_metadata.joblib
Classes: [0 1]


In [3]:
from src.model import train_model

artifacts = train_model(save_artifacts=True)

print("Classes:", artifacts["model"].named_steps["model"].classes_)
print("ROC AUC:", artifacts["metrics"].get("roc_auc"))
print(artifacts["metrics"]["classification_report"])

Arquivo de treino: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/processed/base_processada_reduzida_ML.parquet
Classes do modelo: [0 1]
Pipeline salvo em: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/artifacts/model_pipeline.joblib
Metadados salvos em: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/artifacts/model_metadata.joblib
Classes: [0 1]
ROC AUC: 0.8965706447187929
{'0': {'precision': 0.9898477157360406, 'recall': 0.8024691358024691, 'f1-score': 0.8863636363636364, 'support': 243.0}, '1': {'precision': 0.3684210526315789, 'recall': 0.9333333333333333, 'f1-score': 0.5283018867924528, 'support': 30.0}, 'accuracy': 0.8168498168498168, 'macro avg': {'precision': 0.6791343841838098, 'recall': 0.8679012345679012, 'f1-score': 0.7073327615780446, 'support': 273.0}, 'weighted avg': {'precision': 0.9215590714388471, 'recall': 0.8168498168498168, 'f1-score'

In [4]:
import pandas as pd
from src.model import load_model, predict_dataframe, predict_proba_dataframe

artifacts = load_model()
model = artifacts["model"]

cenario_baixo = pd.DataFrame([{
    "inde": 8.5,
    "n_av": 7,
    "iaa": 8,
    "ieg": 8,
    "ips": 8,
    "ipp": 8,
    "ida": 8,
    "mat": 8,
    "por": 8,
    "ing": 8,
    "ipv": 8,
    "ian": 8,
    "fase_ideal": "7",
}])

cenario_alto = pd.DataFrame([{
    "inde": 4.0,
    "n_av": 3,
    "iaa": 4,
    "ieg": 4,
    "ips": 4,
    "ipp": 4,
    "ida": 4,
    "mat": 3,
    "por": 4,
    "ing": 3,
    "ipv": 4,
    "ian": 4,
    "fase_ideal": "4",
}])

print("Baixo risco:", predict_dataframe(cenario_baixo, model=model, artifacts=artifacts))
print("Baixo risco proba:", predict_proba_dataframe(cenario_baixo, model=model, artifacts=artifacts))

print("Alto risco:", predict_dataframe(cenario_alto, model=model, artifacts=artifacts))
print("Alto risco proba:", predict_proba_dataframe(cenario_alto, model=model, artifacts=artifacts))

Baixo risco: [1]
Baixo risco proba: [[0.0083253 0.9916747]]
Alto risco: [0]
Alto risco proba: [[9.99964692e-01 3.53082007e-05]]


In [7]:
import pandas as pd
from src.model import load_model

artifacts = load_model()
pipeline = artifacts["model"]

preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
coefs = model.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
}).sort_values("coef", ascending=False)

print("Empurram para classe 1")
display(coef_df.head(15))

print("Empurram para classe 0")
display(coef_df.tail(15))

Empurram para classe 1


,feature,coef
11,num__ian,1.818486
19,cat__fase_ideal_Fase 7 (3º EM),1.281161
1,num__n_av,1.267758
12,cat__fase_ideal_ALFA (2º e 3º ano),1.224258
18,cat__fase_ideal_Fase 6 (2º EM),1.022566
10,num__ipv,0.660376
3,num__ieg,0.462437
8,num__por,0.432807
5,num__ipp,0.391018
9,num__ing,0.093376


Empurram para classe 0


,feature,coef
3,num__ieg,0.462437
8,num__por,0.432807
5,num__ipp,0.391018
9,num__ing,0.093376
14,cat__fase_ideal_Fase 2 (5º e 6º ano),0.058329
2,num__iaa,0.013389
0,num__inde,-0.002427
6,num__ida,-0.019919
20,cat__fase_ideal_Fase 8 (Universitários),-0.196947
16,cat__fase_ideal_Fase 4 (9º ano),-0.207049


### 2) Carregar a base de treino


In [8]:
training_path = ROOT_DIR / "data" / "processed" / "base_processada_reduzida_ML.parquet"
df = load_training_data(training_path)

print("Arquivo:", training_path)
print("Shape da base carregada:", df.shape)
display(df.head())


Arquivo: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/processed/base_processada_reduzida_ML.parquet
Shape da base carregada: (3030, 16)


,ra,inde,n_av,iaa,ieg,ips,ipp,ida,mat,por,ing,ipv,ian,fase_ideal,defasagem_ano_seguinte,risco_defasagem_futuro
0,RA-1,5.783000,4.0,8.3,4.1,5.6,NaN,4.0,2.7,3.5,6.0,7.278,5.0,Fase 8 (Universitários),0.0,0
1,RA-1,5.783000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,None,0.0,0
2,RA-1,5.783133,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,None,NaN,<NA>
3,RA-10,5.784000,4.0,8.3,5.2,5.0,NaN,4.1,3.3,2.6,6.4,7.056,5.0,Fase 8 (Universitários),NaN,<NA>
4,RA-100,7.618000,4.0,8.8,7.8,5.0,NaN,7.6,7.0,7.8,8.1,7.250,10.0,Fase 3 (7º e 8º ano),NaN,<NA>


### 3) Entendimento inicial da base

Aqui verificamos:

- colunas disponíveis;
- tipos de dados;
- distribuição do alvo;
- anos presentes;
- valores ausentes.


In [9]:
print("Colunas da base:")
print(df.columns.tolist())

print("\nTipos de dados:")
display(df.dtypes.to_frame("dtype"))

print("\nDistribuição da variável alvo (incluindo nulos):")
display(df[TARGET_COLUMN].value_counts(dropna=False).rename("qtd").to_frame())

print("\nDistribuição percentual do alvo (incluindo nulos):")
display((df[TARGET_COLUMN].value_counts(normalize=True, dropna=False) * 100).round(2).rename("%").to_frame())

if "ano_pede" in df.columns:
    print("\nDistribuição por ano:")
    display(df["ano_pede"].value_counts(dropna=False).sort_index().rename("qtd").to_frame())

print("\nValores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).rename("missing").to_frame())


Colunas da base:
['ra', 'inde', 'n_av', 'iaa', 'ieg', 'ips', 'ipp', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian', 'fase_ideal', 'defasagem_ano_seguinte', 'risco_defasagem_futuro']

Tipos de dados:


,dtype
ra,string[python]
inde,float64
n_av,float64
iaa,float64
ieg,float64
ips,float64
ipp,float64
ida,float64
mat,float64
por,float64



Distribuição da variável alvo (incluindo nulos):


,qtd
risco_defasagem_futuro,
<NA>,1665
0,1213
1,152



Distribuição percentual do alvo (incluindo nulos):


,%
risco_defasagem_futuro,
<NA>,54.95
0,40.03
1,5.02



Valores ausentes por coluna:


,missing
ing,2747
mat,2172
por,2172
fase_ideal,2170
defasagem_ano_seguinte,1665
risco_defasagem_futuro,1665
ipp,1038
ida,178
ipv,178
ips,171


### 4) Preparação das features e do target

A função `prepare_training_features` do novo `src/features.py`:

- seleciona apenas as colunas de entrada do modelo;
- faz coerção dos tipos numéricos e categóricos;
- trata valores ausentes de forma compatível com o pipeline;
- remove registros sem alvo válido para treino.


In [10]:
X, y = prepare_training_features(df, target_column=TARGET_COLUMN)

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

display(X.head())
display(y.head())

print("Distribuição final do target usado no treino:")
display(y.value_counts(dropna=False).sort_index().rename("qtd").to_frame())


Shape de X: (1365, 13)
Shape de y: (1365,)


,inde,n_av,iaa,ieg,ips,ipp,ida,mat,por,ing,ipv,ian,fase_ideal
0,5.7830,4.0,8.3,4.1,5.60,NaN,4.0,2.7,3.5,6.0,7.278,5.0,Fase 8 (Universitários)
1,5.7830,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN
2,7.9162,2.0,8.5,9.4,3.77,6.25,7.0,NaN,NaN,NaN,8.920,10.0,NaN
3,8.1162,2.0,9.0,9.1,7.52,7.50,7.8,NaN,NaN,NaN,9.170,5.0,NaN
4,7.9012,2.0,9.0,9.7,7.52,6.25,7.0,NaN,NaN,NaN,8.920,5.0,NaN


0    0
1    0
2    0
3    0
4    0
Name: risco_defasagem_futuro, dtype: int64

Distribuição final do target usado no treino:


,qtd
risco_defasagem_futuro,
0,1213
1,152


### 5) Estratégia de separação treino e teste

Como este é um problema preditivo com componente temporal, a estratégia preferencial é:

- **treinar com anos anteriores**
- **testar no último ano disponível**

Isso reduz o risco de avaliação otimista demais.

Se não houver anos suficientes com alvo válido, usamos `train_test_split` estratificado como fallback.


In [12]:
from sklearn.model_selection import train_test_split
import pandas as pd

df_trainable = df.loc[y.index].copy()

# Colunas temporais candidatas, por ordem de preferência
temporal_candidates = ["ano_pede", "ano", "year", "ano_referencia"]

temporal_col = next((col for col in temporal_candidates if col in df_trainable.columns), None)

use_temporal_split = False
split_strategy = None

if temporal_col is not None:
    anos_validos = (
        pd.to_numeric(df_trainable[temporal_col], errors="coerce")
        .dropna()
        .astype(int)
        .sort_values()
        .unique()
    )

    use_temporal_split = len(anos_validos) >= 2

if use_temporal_split:
    test_year = int(anos_validos[-1])

    temporal_series = pd.to_numeric(df_trainable[temporal_col], errors="coerce")
    train_mask = temporal_series < test_year
    test_mask = temporal_series == test_year

    X_train = X.loc[train_mask].reset_index(drop=True)
    X_test = X.loc[test_mask].reset_index(drop=True)
    y_train = y.loc[train_mask].reset_index(drop=True)
    y_test = y.loc[test_mask].reset_index(drop=True)

    split_strategy = f"temporal pela coluna '{temporal_col}' (treino < {test_year}, teste = {test_year})"

    # fallback extra caso algum dos lados fique vazio
    if X_train.empty or X_test.empty:
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            stratify=y if y.nunique() > 1 else None,
        )
        split_strategy = "aleatória estratificada (fallback por split temporal inválido)"
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y if y.nunique() > 1 else None,
    )
    split_strategy = "aleatória estratificada (sem coluna temporal)"

print("Estratégia usada:", split_strategy)
print("Treino:", X_train.shape, y_train.shape)
print("Teste :", X_test.shape, y_test.shape)

if temporal_col is not None and temporal_col in X_train.columns and temporal_col in X_test.columns:
    print(f"\nValores de {temporal_col} no treino:")
    display(
        pd.Series(X_train[temporal_col])
        .value_counts(dropna=False)
        .sort_index()
        .rename("qtd")
        .to_frame()
    )

    print(f"\nValores de {temporal_col} no teste:")
    display(
        pd.Series(X_test[temporal_col])
        .value_counts(dropna=False)
        .sort_index()
        .rename("qtd")
        .to_frame()
    )

print("\nDistribuição do target no treino:")
display(y_train.value_counts(dropna=False).sort_index().rename("qtd").to_frame())

print("\nDistribuição do target no teste:")
display(y_test.value_counts(dropna=False).sort_index().rename("qtd").to_frame())


Estratégia usada: aleatória estratificada (sem coluna temporal)
Treino: (1092, 13) (1092,)
Teste : (273, 13) (273,)

Distribuição do target no treino:


,qtd
risco_defasagem_futuro,
0,970
1,122



Distribuição do target no teste:


,qtd
risco_defasagem_futuro,
0,243
1,30


### 6) Treinamento do pipeline

O pipeline é composto por:

- imputação de ausentes nas variáveis numéricas;
- padronização das variáveis numéricas;
- imputação e one-hot encoding das variáveis categóricas;
- regressão logística balanceada.


In [13]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

print("Modelo treinado com sucesso.")
print(pipeline)


Modelo treinado com sucesso.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['inde', 'n_av', 'iaa', 'ieg',
                                                   'ips', 'ipp', 'ida', 'mat',
                                                   'por', 'ing', 'ipv',
                                                   'ian']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),


### 7) Predição e métricas de avaliação

As métricas principais usadas neste notebook são:

- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **ROC AUC** (quando disponível)

Como o objetivo é identificar alunos em risco, **recall** tende a ser especialmente importante, pois mede a capacidade de encontrar casos positivos.


In [14]:
y_pred = pipeline.predict(X_test)

if hasattr(pipeline, "predict_proba"):
    y_score = pipeline.predict_proba(X_test)[:, 1]
else:
    y_score = None

metricas = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
}

if y_score is not None and y_test.nunique() == 2:
    metricas["roc_auc"] = roc_auc_score(y_test, y_score)

metricas_df = pd.DataFrame(
    {"métrica": list(metricas.keys()), "valor": list(metricas.values())}
)
display(metricas_df)

print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Matriz de confusão:")
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Real 0", "Real 1"],
    columns=["Pred 0", "Pred 1"],
)
display(cm)


,métrica,valor
0,accuracy,0.816850
1,precision,0.368421
2,recall,0.933333
3,f1_score,0.528302
4,roc_auc,0.896571


Classification report:
              precision    recall  f1-score   support

           0       0.99      0.80      0.89       243
           1       0.37      0.93      0.53        30

    accuracy                           0.82       273
   macro avg       0.68      0.87      0.71       273
weighted avg       0.92      0.82      0.85       273

Matriz de confusão:


,Pred 0,Pred 1
Real 0,195,48
Real 1,2,28


### 8) Amostra de predições


In [15]:
resultado_teste = X_test.copy()
resultado_teste["y_real"] = y_test.values
resultado_teste["y_pred"] = y_pred

if y_score is not None:
    resultado_teste["prob_risco_1"] = y_score

display(resultado_teste.head(20))


,inde,n_av,iaa,ieg,ips,ipp,ida,mat,por,ing,ipv,ian,fase_ideal,y_real,y_pred,prob_risco_1
301,7.671208,4.0,9.6,8.3,2.52,7.968750,6.8,NaN,NaN,NaN,8.2525,10.0,NaN,0,1,0.935101
235,7.350625,4.0,7.5,8.7,2.52,7.656250,6.5,NaN,NaN,NaN,7.7150,10.0,NaN,0,1,0.907899
139,7.880717,4.0,9.2,9.1,5.00,7.187500,7.3,NaN,NaN,NaN,7.2925,10.0,NaN,0,1,0.820831
432,6.334000,4.0,9.2,7.7,6.90,NaN,4.8,4.3,5.0,5.0,6.2080,2.5,Fase 6 (2º EM),0,0,0.015584
1221,6.243000,4.0,6.7,7.3,5.00,NaN,4.3,5.0,3.5,NaN,5.8330,10.0,Fase 5 (1º EM),0,0,0.274416
156,7.501000,NaN,6.7,NaN,2.52,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,1,1,0.680217
711,7.950000,3.0,8.5,10.0,7.50,NaN,8.8,8.7,9.0,NaN,7.5000,5.0,Fase 2 (5º e 6º ano),0,0,0.019510
1095,8.004000,2.0,9.0,8.8,7.50,NaN,7.5,7.2,7.8,NaN,7.6670,10.0,ALFA (2º e 3º ano),0,0,0.297920
21,7.788875,3.0,0.0,10.0,6.27,9.218750,9.8,NaN,NaN,NaN,8.9500,5.0,NaN,0,0,0.179740
196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,0,1,0.524868


### 9) Interpretação dos coeficientes

Como a regressão logística é interpretável, podemos analisar os coeficientes do modelo treinado.

- coeficientes **positivos** aumentam a chance da classe 1;
- coeficientes **negativos** reduzem a chance da classe 1.

Essa leitura ajuda a explicar o comportamento do modelo para públicos técnicos e de negócio.


In [16]:
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
coeficientes = model.coef_.ravel()

coef_df = pd.DataFrame({
    "feature_transformada": feature_names,
    "coeficiente": coeficientes,
})
coef_df["abs_coeficiente"] = coef_df["coeficiente"].abs()

print("Top coeficientes positivos:")
display(coef_df.sort_values("coeficiente", ascending=False).head(15))

print("Top coeficientes negativos:")
display(coef_df.sort_values("coeficiente", ascending=True).head(15))


Top coeficientes positivos:


,feature_transformada,coeficiente,abs_coeficiente
11,num__ian,1.818486,1.818486
19,cat__fase_ideal_Fase 7 (3º EM),1.281161,1.281161
1,num__n_av,1.267758,1.267758
12,cat__fase_ideal_ALFA (2º e 3º ano),1.224258,1.224258
18,cat__fase_ideal_Fase 6 (2º EM),1.022566,1.022566
10,num__ipv,0.660376,0.660376
3,num__ieg,0.462437,0.462437
8,num__por,0.432807,0.432807
5,num__ipp,0.391018,0.391018
9,num__ing,0.093376,0.093376


Top coeficientes negativos:


,feature_transformada,coeficiente,abs_coeficiente
13,cat__fase_ideal_Fase 1 (4º ano),-1.844619,1.844619
15,cat__fase_ideal_Fase 3 (7º e 8º ano),-0.889955,0.889955
7,num__mat,-0.729818,0.729818
17,cat__fase_ideal_Fase 5 (1º EM),-0.500155,0.500155
4,num__ips,-0.288615,0.288615
16,cat__fase_ideal_Fase 4 (9º ano),-0.207049,0.207049
20,cat__fase_ideal_Fase 8 (Universitários),-0.196947,0.196947
6,num__ida,-0.019919,0.019919
0,num__inde,-0.002427,0.002427
2,num__iaa,0.013389,0.013389


### 10) Salvamento dos artefatos

Ao final, o notebook salva:

- o pipeline treinado;
- os metadados necessários para inferência no app Streamlit.


In [17]:
artifacts_dir = ROOT_DIR / "data" / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, MODEL_FILE)
joblib.dump(
    {
        "feature_columns": FEATURE_COLUMNS,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "target_column": TARGET_COLUMN,
        "metrics": {
            **metricas,
            "split_strategy": split_strategy,
            "train_rows": int(len(X_train)),
            "test_rows": int(len(X_test)),
        },
    },
    METADATA_FILE,
)

print("Modelo salvo em:", MODEL_FILE)
print("Metadados salvos em:", METADATA_FILE)


Modelo salvo em: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/artifacts/model_pipeline.joblib
Metadados salvos em: /home/sergioraulino/data-science/fiap/postech-FIAP/FASE5/fiap-datathon-passos-magicos/data/artifacts/model_metadata.joblib


## Conclusão

Com a nova base de modelagem e o novo `src/features.py`, este notebook passa a treinar um modelo **preditivo de fato**, pois utiliza informações do ano atual para estimar o risco de defasagem no ano seguinte.

Os principais ganhos desta abordagem são:

- eliminação do vazamento de informação do alvo atual;
- maior aderência ao problema de negócio;
- avaliação mais coerente com cenário real, especialmente com separação temporal;
- pipeline pronto para serialização e deploy no Streamlit.
